# Composition across database and outcome

**Producer for `results/composition_across_db_and_outcome.json`,** the last of the
three results that had no code in the repo. It carries §02's claim that *which
health system the data came from* explains more of the decomposition's shape than
*which outcome is being predicted* — 50.1% against 33.3% for unique-to-value.

## Design

A balanced **2 databases x 4 outcomes x 5 seeds** layout on the matched
12-analyte panel at k=13, BROJA. Because it is balanced, each atom's variation
in *share* can be attributed by ordinary two-way ANOVA to:

| source | meaning |
|---|---|
| database | MIMIC-IV vs eICU-CRD |
| outcome | mortality / sex / age / race |
| interaction | the effect of database differing by outcome |
| seed | residual k-means clustering noise |

Seeds are the replicates, so the residual term is exactly the discretisation
instability the k-sweep measures — which means the database and outcome effects
can be read as real only insofar as they exceed it.

The atom shares are recomputed here per seed rather than read from
`headline_matched_panel12.json`, which stores only means and SDs. Same protocol,
same seeds, so the per-cell means reproduce that file.

In [1]:
import sys
sys.path.insert(0, "../src")

import json, time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import mimic_channels as mimic
import eicu_channels as eicu
import pid_broja as pb
import quantize as qz

pd.set_option("display.width", 220)

K = 13
SEEDS = [0, 1, 2, 3, 4]
ATOMS = ("red", "u_val", "u_str", "syn")
TARGETS = ["mortality", "race_binary", "gender_female", "age_ge65"]

MG_MIMIC = 50960
PANEL12_EICU = {"bicarbonate", "chloride", "creatinine", "glucose", "Hct", "Hgb",
                "platelets x 1000", "potassium", "RBC", "sodium", "BUN", "WBC x 1000"}

cc_m = mimic.complete_case(itemid_scope=set(mimic.CORE) - {MG_MIMIC})
cc_e = eicu.complete_case(labname_scope=PANEL12_EICU)
DBS = [("MIMIC-IV", mimic, cc_m), ("eICU-CRD", eicu, cc_e)]
print(f"balanced {len(DBS)} x {len(TARGETS)} x {len(SEEDS)} = "
      f"{len(DBS)*len(TARGETS)*len(SEEDS)} decompositions at k={K}")

balanced 2 x 4 x 5 = 40 decompositions at k=13


## 1. Per-seed atom shares

In [2]:
# SHARED CODEBOOK (2026-09-01): one k-means fit per (database, seed) on the FULL
# cohort, masked to each target's rows, so every outcome is described in the same
# vocabulary. Must match notebooks/02 -- a floor or a variance split measured
# under a different discretisation is not comparable to the headline it refers to.
CODEBOOK, DEMO = {}, {}
for _db, _mod, _cc in DBS:
    DEMO[_db] = _mod.demographic_targets()[0]
    for _sd in SEEDS:
        _v, _ = qz.kmeans_levels(qz.standardize_for_kmeans(_cc.Xv), K, seed=_sd)
        _s, _ = qz.kmeans_levels(
            qz.standardize_for_kmeans(_cc.Xs, sentinel=_mod.TIME_SENTINEL), K, seed=_sd)
        CODEBOOK[(_db, _sd)] = (_v, _s)

rows = []
t0 = time.time()
for db, mod, cc in DBS:
    demo = DEMO[db]
    for target in TARGETS:
        ch = cc if target == "mortality" else mod.retarget(cc, demo[target], target)
        keep = (None if target == "mortality"
                else demo[target].reindex(pd.Index(cc.groups)).notna().to_numpy())
        for sd in SEEDS:
            v, s = CODEBOOK[(db, sd)]
            if keep is not None: v, s = v[keep], s[keep]
            d = pb.decompose(v, s, ch.y, measure="broja")
            if not d["converged"]:
                print(f"  !! {db}/{target}/seed{sd} did not converge -- excluded")
                continue
            a = d["atoms_pct_of_joint"]
            rows.append({"db": db, "target": target, "seed": sd,
                         **{k: a[k] for k in ATOMS},
                         "non_value": 100.0 - a["u_val"]})
df = pd.DataFrame(rows)
print(f"{len(df)} cells in {time.time()-t0:.0f}s   "
      f"(balanced: {df.groupby(['db','target']).size().nunique() == 1})")
print("\nper-cell means (should match results/headline_matched_panel12.json)\n")
print(df.groupby(["db","target"])[list(ATOMS)].mean()
        .to_string(float_format=lambda x: f"{x:.2f}"))

40 cells in 3s   (balanced: True)

per-cell means (should match results/headline_matched_panel12.json)

                         red  u_val  u_str   syn
db       target                                 
MIMIC-IV age_ge65      11.88  61.35  -0.00 26.78
         gender_female 18.15  45.42   0.09 36.34
         mortality     24.54  39.78   4.16 31.51
         race_binary   23.09  32.95  12.52 31.44
eICU-CRD age_ge65       9.25  81.46  -0.00  9.29
         gender_female  1.82  87.67  -0.00 10.52
         mortality     19.97  49.10   0.14 30.79
         race_binary    9.89  69.39  -0.00 20.72


## 2. Variance decomposition

Two-way ANOVA with replication. Sums of squares are computed directly rather
than via a formula library, so the balanced-design assumption is explicit.

In [3]:
def two_way(df, col):
    """SS decomposition for a balanced a x b design with n replicates."""
    grand = df[col].mean()
    a, b = df.db.nunique(), df.target.nunique()
    n = len(df) // (a * b)
    ss_tot = ((df[col] - grand) ** 2).sum()
    ss_db = b * n * ((df.groupby("db")[col].mean() - grand) ** 2).sum()
    ss_out = a * n * ((df.groupby("target")[col].mean() - grand) ** 2).sum()
    cellm = df.groupby(["db", "target"])[col].mean()
    dbm = df.groupby("db")[col].mean(); outm = df.groupby("target")[col].mean()
    ss_int = n * sum((cellm[(i, j)] - dbm[i] - outm[j] + grand) ** 2
                     for i in dbm.index for j in outm.index)
    ss_res = ss_tot - ss_db - ss_out - ss_int
    df_db, df_out, df_int, df_res = a - 1, b - 1, (a - 1) * (b - 1), a * b * (n - 1)
    ms_res = ss_res / df_res
    return {
        "database": round(100 * ss_db / ss_tot, 1),
        "outcome": round(100 * ss_out / ss_tot, 1),
        "interaction": round(100 * ss_int / ss_tot, 1),
        "seed_noise": round(100 * ss_res / ss_tot, 1),
        "p_db": float(stats.f.sf((ss_db / df_db) / ms_res, df_db, df_res)),
        "p_outcome": float(stats.f.sf((ss_out / df_out) / ms_res, df_out, df_res)),
        "p_interaction": float(stats.f.sf((ss_int / df_int) / ms_res, df_int, df_res)),
    }

var_dec = {a: two_way(df, a) for a in ATOMS}
print("share of each atom's total variation, %\n")
print(pd.DataFrame(var_dec).T[["database","outcome","interaction","seed_noise"]]
      .to_string(float_format=lambda x: f"{x:.1f}"))
print("\np-values\n")
print(pd.DataFrame(var_dec).T[["p_db","p_outcome","p_interaction"]]
      .to_string(float_format=lambda x: f"{x:.2e}"))

share of each atom's total variation, %

       database  outcome  interaction  seed_noise
red        37.2     44.0         14.5         4.3
u_val      50.9     33.6         11.9         3.7
u_str      22.0     33.0         33.0        11.9
syn        48.2     23.0         21.8         7.0

p-values

          p_db  p_outcome  p_interaction
red   2.39e-17   6.49e-17       2.15e-10
u_val 2.52e-20   3.40e-16       3.79e-10
u_str 9.04e-09   2.34e-09       2.33e-09
syn   7.18e-16   3.28e-10       6.48e-10


## 3. Main effects and the interaction

In [4]:
db_eff = df.groupby("db")[list(ATOMS) + ["non_value"]].mean().round(2)
out_eff = df.groupby("target")[list(ATOMS) + ["non_value"]].mean().round(2)
cell = df.groupby(["db","target"])[list(ATOMS) + ["non_value"]].mean()
gap = (cell.loc["MIMIC-IV"] - cell.loc["eICU-CRD"]).round(1)

print("database main effect\n"); print(db_eff.to_string())
print("\noutcome main effect (ordered by how much is NOT explained by values alone)\n")
print(out_eff.sort_values("non_value", ascending=False).to_string())
print("\nper-outcome database gap (MIMIC minus eICU)\n"); print(gap.to_string())

agree = gap.u_val.abs().idxmin(); disagree = gap.u_val.abs().idxmax()
print(f"\n  databases agree most on {agree} (u_val gap {gap.u_val[agree]:+.1f}, "
      f"syn {gap.syn[agree]:+.1f})")
print(f"  databases disagree most on {disagree} (u_val gap {gap.u_val[disagree]:+.1f}, "
      f"syn {gap.syn[disagree]:+.1f})")
print(f"  seed noise spans {min(v['seed_noise'] for v in var_dec.values()):.1f}%"
      f" to {max(v['seed_noise'] for v in var_dec.values()):.1f}% of total variation")

database main effect

            red  u_val  u_str    syn  non_value
db                                             
MIMIC-IV  19.41  44.88   4.19  31.52      55.12
eICU-CRD  10.23  71.90   0.04  17.83      28.10

outcome main effect (ordered by how much is NOT explained by values alone)

                 red  u_val  u_str    syn  non_value
target                                              
mortality      22.25  44.44   2.15  31.15      55.56
race_binary    16.49  51.17   6.26  26.08      48.83
gender_female   9.98  66.55   0.04  23.43      33.45
age_ge65       10.56  71.40  -0.00  18.03      28.60

per-outcome database gap (MIMIC minus eICU)

                red  u_val  u_str   syn  non_value
target                                            
age_ge65        2.6  -20.1   -0.0  17.5       20.1
gender_female  16.3  -42.2    0.1  25.8       42.2
mortality       4.6   -9.3    4.0   0.7        9.3
race_binary    13.2  -36.4   12.5  10.7       36.4

  databases agree most on mortality (u

## 4. Save

In [ ]:
# Numbers only, same rule as notebooks 04 and 05: what the variance split MEANS
# is argued in the manuscript, not stored beside the numbers where it silently
# goes out of date the next time either is re-run.
#
# `summary` holds the facts the old prose findings asserted -- which outcomes the
# databases agree and disagree on, the ordering by non-value share -- read off
# the numbers rather than restated in sentences.
order = out_eff.sort_values("non_value", ascending=False)

payload = {
    "design": (f"Both databases on the same 12 analytes (24-d channels), k={K} fixed pre-collapse "
               f"grid, shared codebook, BROJA, {len(SEEDS)} k-means seeds per cell. Balanced "
               f"{len(DBS)} databases x {len(TARGETS)} outcomes x {len(SEEDS)} seeds, so variation "
               f"in each atom's SHARE is attributable to database, outcome, their interaction, or "
               f"seed noise by two-way ANOVA with seeds as replicates."),
    "variance_decomposition_pct_of_total": var_dec,
    "database_main_effect": {k: v for k, v in db_eff.T.to_dict().items()},
    "outcome_main_effect": {k: v for k, v in out_eff.T.to_dict().items()},
    "per_outcome_database_gap_MIMIC_minus_eICU": {k: v for k, v in gap.T.to_dict().items()},
    "summary": {
        "outcomes_by_non_value_share_desc": list(order.index),
        "databases_agree_most_on": agree,
        "databases_disagree_most_on": disagree,
        "u_val_gap_at_agreement": float(gap.u_val[agree]),
        "u_val_gap_at_disagreement": float(gap.u_val[disagree]),
        "seed_noise_pct_min": float(min(v["seed_noise"] for v in var_dec.values())),
        "seed_noise_pct_max": float(max(v["seed_noise"] for v in var_dec.values())),
    },
    "per_seed": rows,
}
out = Path("../results/composition_across_db_and_outcome.json")
out.write_text(json.dumps(payload, indent=2, default=float))
print(f"wrote {out}  ({len(rows)} per-seed records, numbers only)")


wrote ../results/composition_across_db_and_outcome.json  (40 per-seed records, numbers only)
